In [ ]:
!git clone https://github.com/Airbwender/pixelate.git /kaggle/working/pixelate
%cd /kaggle/working/pixelate

!pip install -q -r requirements.txt
!pip install -q -U "torchao>=0.16.0" "peft>=0.20.0" bitsandbytes

In [ ]:
import torch
import torchao
import peft

print("cuda:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print(
        "vram:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("torch:", torch.__version__)
print("torchao:", torchao.__version__)
print("peft:", peft.__version__)

In [ ]:
# dataset check
from pathlib import Path

dataset_dir = Path(
    "/kaggle/input/datasets/airbwender/anime-pixel-art-2k/anime-pixel-art-2k"
)

assert dataset_dir.exists(), f"dataset not found: {dataset_dir}"
assert (dataset_dir / "metadata.jsonl").exists(), "metadata.jsonl not found"

image_count = len(
    list((dataset_dir / "images").glob("*.png"))
)

assert image_count == 2000, (
    f"expected 2000 images, found {image_count}"
)

print("dataset:", dataset_dir)
print("images:", image_count)
print("metadata: ok")

In [ ]:
from pathlib import Path

path = Path("training.py")
text = path.read_text()

text = text.replace(
    'DATASET_DIR = "data/anime-pixel-art-2k"',
    'DATASET_DIR = "/kaggle/input/datasets/airbwender/anime-pixel-art-2k/anime-pixel-art-2k"'
)

if "unet.enable_gradient_checkpointing()" not in text:
    text = text.replace(
        "unet.add_adapter(lora_config)",
        "unet.add_adapter(lora_config)\n\n    unet.enable_gradient_checkpointing()"
    )

if "bnb.optim.AdamW8bit" not in text:
    text = text.replace(
        "optimizer = torch.optim.AdamW(",
        "import bitsandbytes as bnb\n\n    optimizer = bnb.optim.AdamW8bit("
    )

path.write_text(text)

print("training.py configured")
print(
    "dataset:",
    [x for x in text.splitlines() if x.startswith("DATASET_DIR")][0]
)
print(
    "steps:",
    [x for x in text.splitlines() if x.startswith("MAX_TRAIN_STEPS")][0]
)
print(
    "gradient checkpointing:",
    "unet.enable_gradient_checkpointing()" in text
)
print(
    "8-bit optimizer:",
    "bnb.optim.AdamW8bit" in text
)

In [ ]:
!python -c "import training; print('training.py imports successfully')"

In [ ]:
%cd /kaggle/working/pixelate

!python training.py

In [ ]:
!find outputs -maxdepth 3 -type f